In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime
from ICONProcessor import ICONDataGrid

data_dir = '../data_in/'
data_out_dir = '../data_out/'

In [ ]:
def get_file(avg=False, sfc=False, pl=False):
    if avg:
        avg_str = '_30min'
    else:
        avg_str = ''
    
    if pl:
        suffix = 'pl__uv'
    elif sfc:
        suffix = 'ml__sfc'
    else:
        suffix = 'ml__lowest5'
    
      
    file = f'{project_root}/{output_dir}/{exp}_LES{avg_str}_51m_{suffix}.nc'
    return file

def format_datetime64(val, format_str='%Y-%m-%d %H:%M:%S'):
    dt = pd.to_datetime(str(val))
    return dt.strftime(format_str)

def get_daterange_str(ds):
    t = pd.to_datetime(ds.time.values,utc=True)
    return f'{t[0].strftime("%Y%m%d")}-{t[-1].strftime("%Y%m%d")}'

## Script settings

In [ ]:
scope = 'H2'

project_root = '/work/bb1461/'
exp = 'v370_2030'

In [ ]:
if scope == 'H2':
    scope_name = 'HEFEX II'
    output_dir = 'cdo_scripts/output_hefex2/'
    experiment_id =f"hefex2/{exp}/exp_R3B15_51m"
elif scope == 'H3':
    scope_name = 'HEFEX III'
    output_dir = 'cdo_scripts/output_hefex3/'
    experiment_id = f"hefex3/{exp}/exp_R3B15_51m"
else:
    print('Invalid scope!')

loc_file = data_dir + f'{scope}_Coords_w_cells.csv'

## Load and prepare location dimension

In [ ]:
df_locs = pd.read_csv(loc_file).sort_values('topo')
df_locs = df_locs[~df_locs['cell_id'].isna()]
df_locs

In [ ]:
dim_loc = df_locs['New Name']
dim_loc = dim_loc.replace('Hintereis','STHE')

dim_loc_lat = np.array(df_locs['Lat'])
dim_loc_lon = df_locs['Lon']
dim_loc_cid = df_locs['cell_id'].astype(int)
dim_loc_ele = df_locs['topo']

## Load and prepare ICON data for locations

In [ ]:
def create_data_array(ds, icon_var, height, name, time_dim='time'):
    # get time from dataset
    time = [ICONDataGrid.convert_float_dt(f) for f in ds.time.values]
    time_naive = [dt.replace(tzinfo=None) for dt in time]
    if time_dim.endswith('_avg'):
        t_res = '30min averages (origin is the last value of the timeseries)'
    else:
        t_res = 'Hourly instantaneous values'

    # get data_in for variable (all cell IDs)
    data = ds[icon_var].values[:, height, np.array(dim_loc_cid-1)]

    # create DataArray
    xr_data = xr.DataArray(
        data,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            "location": ("location", dim_loc, {"long_name": "Location name from field campaign"}),
            "lat": ("location", dim_loc_lat, {"long_name": "Latitude", "units": "degrees_north"}),
            "lon": ("location", dim_loc_lon, {"long_name": "Longitude", "units": "degrees_east"}),
            "cid": ("location", dim_loc_cid, {"long_name": "ICON Cell ID of location"}),
            "ele": ("location", dim_loc_ele, {"long_name": "Elevation on ICON grid", "units": "meters"})
        },
        dims=[time_dim, "location"],
        name=name,
        attrs={
            "long_name": ds[icon_var].long_name,
            "standard_name": ds[icon_var].standard_name,
            "units": ds[icon_var].units,
            "temporal_resolution": t_res,
        }
    )
    return xr_data

In [ ]:
# surface values
ds = xr.open_dataset(get_file(sfc=True))
xr_t_2m  = create_data_array(ds, 't_2m',  0, 't_2m')
xr_qv_2m = create_data_array(ds, 'qv_2m', 0, 'qv_2m')

# values from lowest model level 
ds = xr.open_dataset(get_file(sfc=False))
xr_t_5m  = create_data_array(ds, 'temp', -1, 't_5m')
xr_qv_5m = create_data_array(ds, 'qv',   -1, 'qv_5m')
xr_u_5m  = create_data_array(ds, 'u',    -1, 'u_5m')
xr_v_5m  = create_data_array(ds, 'v',    -1, 'v_5m')

# values from 500hPa pressure level
ds = xr.open_dataset(get_file(pl=True))
xr_u_500hPa  = create_data_array(ds, 'u', 0, 'u_500hPa')
xr_v_500hPa  = create_data_array(ds, 'v', 0, 'v_500hPa')

# 30min avg values
ds = xr.open_dataset(get_file(avg=True))
xr_u_5m_avg  = create_data_array(ds, 'u', -1, 'u_5m_avg', 'time_avg')
xr_v_5m_avg  = create_data_array(ds, 'v', -1, 'v_5m_avg', 'time_avg')

## Build dataset

In [ ]:
# Combine DataArrays
ds_out = xr.Dataset({
    't_2m' : xr_t_2m,
    't_5m' : xr_t_5m,
    'qv_2m': xr_qv_2m,                     
    'qv_5m': xr_qv_5m, 
    'u_5m' : xr_u_5m,
    'v_5m' : xr_v_5m,
    'u_500hPa': xr_u_500hPa,                     
    'v_500hPa': xr_v_500hPa, 
    'u_5m_avg' : xr_u_5m_avg,
    'v_5m_avg' : xr_v_5m_avg,
})

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="1h")
ds_out = ds_out.reindex(time=time_full)

ds_out = ds_out.sortby("time_avg")
ds_out = ds_out.sel(time_avg=~ds_out.indexes["time_avg"].duplicated())
time_full = pd.date_range(ds_out.time_avg.min().values, ds_out.time_avg.max().values, freq="30min")
ds_out = ds_out.reindex(time_avg=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Sliced ICON-LES dataset (51m resolution) for {scope_name} locations",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "experiment_id": experiment_id,
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Experiment based on glacier outlines from OGGM SSP370, year 2030"
}

# Write
output_file = data_out_dir + f'HEFEX{scope[1]}_ICON_{exp}_aws_locations_1h_avg30min_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

In [ ]:
ds_out